# MySQL 入库清单与数据核对

从八个清洗阶段的产物中选出九张拟入库表。先在 Pandas 中恢复类型和验证主外键口径，下一课再将验证通过的表写成 Parquet。

本课使用订单 v2。订单商品、支付和评价分别保留各自的明细粒度。


## 1. 查看入库来源、粒度和主键

清单记录了每张表的来源、预计行数、主键、字段类型和允许为空的字段。


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from mysql_input import read_manifest, load_input_table, check_input_relationships

manifest = read_manifest()
table_specs = manifest['tables']
table_plan = pd.DataFrame([
    {
        '拟入库表': name,
        '来源': spec['source'],
        '一行的含义': spec['grain'],
        '主键': ' + '.join(spec['primary_key']),
        '预计行数': spec['expected_rows'],
    }
    for name, spec in table_specs.items()
])
display(table_plan)


,拟入库表,来源,一行的含义,主键,预计行数
0,customers,customers_v1/customers_clean_v1.csv,一个订单关联的客户记录,customer_id,99441
1,sellers,sellers_v1/sellers_clean_v1.csv,一个卖家,seller_id,3095
2,category_translation,products_v1/category_translation_v1.csv,一个原始品类的英文映射,product_category_name,71
3,products,products_v1/products_clean_v1.csv,一个商品,product_id,32951
4,geo_reference,geolocation_v1/geo_reference_v1.csv,一个邮编前缀与州组合的代表坐标,geolocation_zip_code_prefix + geolocation_state,19019
5,orders,orders_v2/orders_clean_v2.csv,一笔订单,order_id,99441
6,order_items,items_v1/items_clean_v1.csv,一笔订单中的一条商品明细,order_id + order_item_id,112650
7,order_payments,payments_v1/payments_clean_v1.csv,一笔订单的一条支付记录,order_id + payment_sequential,103886
8,order_reviews,reviews_v1/reviews_clean_v1.csv,一笔订单的一条评价记录,order_id + review_id,99224


## 2. 恢复类型并检查每张表

CSV 保存的是文本，重新读取后根据清单恢复日期、整数和布尔标记。
标识与邮编保留字符串。金额用整数分字段核对。清单允许为空的字段保留原有缺失状态。


In [2]:
processed_dir = project_dir / 'data' / 'processed'
tables = {
    name: load_input_table(processed_dir, name, spec)
    for name, spec in table_specs.items()
}
table_checks = pd.DataFrame([
    {
        '表': name,
        '实际行数': len(frame),
        '行数差额': len(frame) - table_specs[name]['expected_rows'],
        '主键缺失行数': frame[table_specs[name]['primary_key']].isna().any(axis=1).sum(),
        '主键重复行数': frame.duplicated(table_specs[name]['primary_key']).sum(),
    }
    for name, frame in tables.items()
])
display(table_checks)

# 观察几种代表字段的目标类型与实际恢复出的 Pandas 类型。
samples = [
    ('customers', 'customer_zip_code_prefix'),
    ('orders', 'order_purchase_timestamp'),
    ('order_items', 'price_cents'),
    ('products', 'flag_weight_zero'),
    ('geo_reference', 'representative_latitude'),
]
display(pd.DataFrame([
    {
        '字段': name + '.' + column,
        '拟用 MySQL 类型': table_specs[name]['columns'][column],
        'Pandas 类型': str(tables[name][column].dtype),
        '缺失数': tables[name][column].isna().sum(),
    }
    for name, column in samples
]))


,表,实际行数,行数差额,主键缺失行数,主键重复行数
0,customers,99441,0,0,0
1,sellers,3095,0,0,0
2,category_translation,71,0,0,0
3,products,32951,0,0,0
4,geo_reference,19019,0,0,0
5,orders,99441,0,0,0
6,order_items,112650,0,0,0
7,order_payments,103886,0,0,0
8,order_reviews,99224,0,0,0


,字段,拟用 MySQL 类型,Pandas 类型,缺失数
0,customers.customer_zip_code_prefix,CHAR(5),string,0
1,orders.order_purchase_timestamp,DATETIME,datetime64[ns],0
2,order_items.price_cents,BIGINT,Int64,0
3,products.flag_weight_zero,TINYINT,boolean,2
4,geo_reference.representative_latitude,DOUBLE,float64,0


## 3. 区分必备关联与覆盖检查

订单到客户、明细到订单等业务关联计划设置外键，检查缺失键和非空未匹配键。
品类翻译和代表坐标允许有覆盖缺口。后续查这些补充属性时使用左关联。


In [3]:
relationship_checks = check_input_relationships(tables, manifest['relationships'])
display(relationship_checks)


,来源表,键,目标表,规则,键缺失,键非空但未匹配,已匹配
0,orders,customer_id,customers,拟设外键,0,0,99441
1,order_items,order_id,orders,拟设外键,0,0,112650
2,order_items,product_id,products,拟设外键,0,0,112650
3,order_items,seller_id,sellers,拟设外键,0,0,112650
4,order_payments,order_id,orders,拟设外键,0,0,103886
5,order_reviews,order_id,orders,拟设外键,0,0,99224
6,products,product_category_name,category_translation,覆盖检查,610,13,32328
7,customers,customer_zip_code_prefix + customer_state,geo_reference,覆盖检查,0,279,99162
8,sellers,seller_zip_code_prefix + seller_state,geo_reference,覆盖检查,0,42,3053


## 4. 核对金额与缺失值

把清洗阶段的整数分合计与本次读入的合计对照。
对可空标记分别数 True、False 和缺失，观察三种状态是否保留。


In [4]:
amount_checks = []
for name, spec in table_specs.items():
    for column, expected in spec.get('expected_totals', {}).items():
        actual = int(tables[name][column].sum())
        amount_checks.append({
            '字段': name + '.' + column,
            '基准金额（分）': expected,
            '读入金额（分）': actual,
            '差额（分）': actual - expected,
        })

display(pd.DataFrame(amount_checks))
display(tables['products']['flag_weight_zero'].value_counts(dropna=False))


,字段,基准金额（分）,读入金额（分）,差额（分）
0,order_items.price_cents,1359164370,1359164370,0
1,order_items.freight_cents,225190954,225190954,0
2,order_payments.payment_value_cents,1600887212,1600887212,0


flag_weight_zero
False    32945
True         4
<NA>         2
Name: count, dtype: Int64

## 我的理解

逐格运行并核对九张表的主键和金额。请发来第 3 格的关联检查表，再回答两题。

1. 为什么订单到客户用 customer_id 关联？customer_unique_id 在之后的复购分析里起什么作用？
因为一个客户可能有多个订单,而customer_id唯一对应一个order_id；可以看一个客户购买了多少订单。
2. 商品品类非空但在翻译表里找不到时，为什么适合用左关联获取译名？如果强制要求品类必须在翻译表中，会影响哪些记录？
因为左关联对右表中没有查询到的行会保留而不是直接删除，这样关联之后可以人工翻译在翻译表里找不到的商品品类；会影响品类非空但在翻译表中找不到的品类。
下一课把验证通过的表写入 Parquet，再重新读取并比较类型、行数和金额。
